321 Assignment 3 - Public Ciphers

William Kim and James Irwin 

## Task 1: Implement Diffie Hellman Key Exchange

In [4]:
from Crypto.Hash import SHA256 
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from Crypto.Util.Padding import pad, unpad
from Crypto.Util.number import getPrime
import secrets
import hashlib
from math import gcd

In [41]:
#Helper Functions. Truncating the output of SHA256 to 16 bytes, and working accordingly 
def derive_aes_key(shared_secret):
    hash_obj = SHA256.new(str(shared_secret).encode())
    return hash_obj.digest()[:16]


def aes_encrypt(key, iv, message):
    cipher = AES.new(key, AES.MODE_CBC, iv)
    ciphertext = cipher.encrypt(pad(message.encode(), 16))
    return ciphertext


def aes_decrypt(key, iv, ciphertext):
    cipher = AES.new(key, AES.MODE_CBC, iv)
    plaintext = unpad(cipher.decrypt(ciphertext), 16)
    return plaintext.decode()

We will run Diffie-Hellman on a small scale for Alice and Bob, to demonstrate how it works

In [42]:
# Bob and Alice Keys Small setting example
q = 37
a = 5

# Alice
priv_alice = secrets.randbelow(q) #Alice private
pub_alice = pow(a,priv_alice, q) #Alice public

# Bob
priv_bob = secrets.randbelow(q) #Bob private
pub_bob = pow(a, priv_bob, q) #Bob public

print("Alice public value: ", pub_alice)
print("Bob public value: ", pub_bob)


Alice public value:  20
Bob public value:  21


We can construct a shared value for each party, using your private value, and a public value from someone else, and these shared values should be the same. 
    

In [43]:
shared_alice = pow(pub_bob, priv_alice, q)
shared_bob = pow(pub_alice, priv_bob, q)

print("Alice's secret value:", shared_alice)
print("Bob's secret value:  ", shared_bob)
print("Secrets match:", shared_alice == shared_bob)

Alice's secret value: 4
Bob's secret value:   4
Secrets match: True


Key derivation: SHA-256 to get symmetric encryption key 

In [44]:
aes_key = derive_aes_key(shared_alice)
i_v = b'\x00' * 16  # shared initialization vector

c0 = aes_encrypt(aes_key, i_v, "Hello Bob")
print("Alice encrypts the message 'Hello Bob', to the ciphertext: ", c0)

Alice encrypts the message 'Hello Bob', to the ciphertext:  b'O\x14\x8b \x1f\x96er\x05\xb4g\xaf\xe3D\xff\x13'


Bob will now take Alices encrypted cipher text, and decrypt it. 

In [45]:
decrypted_aes_bob = aes_decrypt(aes_key, i_v, c0)
print ("Bob decrypts the message from the ciphertext : ", decrypted_aes_bob)

Bob decrypts the message from the ciphertext :  Hello Bob


Bob will then encrypt his own message to send back 

In [46]:
c1 = aes_encrypt(aes_key, i_v, "Hi there Alice!")

print("Bob's message 'Hi there Alice' will be encrypted to :", c1)

Bob's message 'Hi there Alice' will be encrypted to : b'\xec`[V\xa4\xb5^\xe7\x1c\xf4\xc4\x84\xbd*~\x00'


Alice will now receive Bob's encrypted data, and decrypt it to receive the message

In [47]:
decrypted_aes_alice = aes_decrypt(aes_key, i_v, c1)
print ("Alice decrypts the message from the ciphertext : ", decrypted_aes_alice)

Alice decrypts the message from the ciphertext :  Hi there Alice!


Trying it with some "real numbers", as defined in the assignment spec.

In [2]:

q_hex = """
B10B8F96A080E01DDE92DE5EAE5D54EC52C99FBCFB06A3C6
9A6A9DCA52D23B616073E28675A23D189838EF1E2EE652C0
13ECB4AEA906112324975C3CD49B83BFACCBDD7D90C4BD70
98488E9C219A73724EFFD6FAE5644738FAA31A4FF55BCCC0
A151AF5F0DC8B4BD45BF37DF365C1A65E68CFDA76D4DA708
DF1FB2BC2E4A4371
"""


a_hex = """
A4D1CBD5C3FD34126765A442EFB99905F8104DD258AC507F
D6406CFF14266D31266FEA1E5C41564B777E690F5504F213
160217B4B01B886A5E91547F9E2749F4D7FBD7D3B9A92EE1
909D0D2263F80A76A6A24C087A091F531DBF0A0169B6A28A
D662A4D18E73AFA32D779D5918D08BC8858F4DCEF97C2A24
855E6EEB22B3B2E5
"""

q = int(q_hex.replace("\n", "").replace(" ", ""), 16)
a = int(a_hex.replace("\n", "").replace(" ", ""), 16)

Alice and Bob choose some random 256 bit private keys.

In [5]:
alice_priv = int.from_bytes(get_random_bytes(32), 'big')
bob_priv = int.from_bytes(get_random_bytes(32), 'big')

Compute public keys

In [6]:
alice_pub = pow(a, alice_priv, q)
bob_pub = pow(a, bob_priv, q)

Finally check to see if both sides compute the same value

In [7]:
alice_shared = pow(bob_pub, alice_priv, q)
bob_shared = pow(alice_pub, bob_priv, q)

print("Do they compute the same shared key?", alice_shared == bob_shared)

Do they compute the same shared key? True


## Task 2: Implement MITM Key Fixing & Negotiated Groups

### Modify Task 1 Implementation

In [49]:
# Alice sends q and a to Bob
q = 37
a = 5

# Alice
priv_alice = secrets.randbelow(q) #Alice private
pub_alice = pow(a,priv_alice, q) #Alice public

# Bob
priv_bob = secrets.randbelow(q) #Bob private
pub_bob = pow(a, priv_bob, q) #Bob public

print("Alice public value: ", pub_alice)
print("Bob public value: ", pub_bob)

Alice public value:  29
Bob public value:  19


In [50]:
# at this point, Mallory intercepts and sends q to both Alice and Bob
mallory_q = 37

print("Alice public value: ", mallory_q)
print("Bob public value: ", mallory_q)

Alice public value:  37
Bob public value:  37


In [51]:
# now, the shared key is calculated using q instead of public keys, 
# resulting in a secret value of 0

shared_alice = pow(mallory_q, priv_alice, q)
shared_bob = pow(mallory_q, priv_bob, q)

print("Alice's secret value:", shared_alice)
print("Bob's secret value:  ", shared_bob)
print("Secrets match:", shared_alice == shared_bob)

Alice's secret value: 0
Bob's secret value:   0
Secrets match: True


In [52]:
# Mallory knows that math will work out this way
mallory_shared = 0

# alice generates key
aes_key = derive_aes_key(shared_alice)
i_v = b'\x00' * 16  # shared initialization vector

# mallory is able to generate the same key
mallory_aes_key = derive_aes_key(mallory_shared)

In [53]:
# and then decrypt messages! 

# alice attempts to send message
c0 = aes_encrypt(aes_key, i_v, "Hello Bob")
print("Alice encrypts the message 'Hello Bob', to the ciphertext: ", c0)

# mallory decrypts
decrypted_aes_mallory = aes_decrypt(mallory_aes_key, i_v, c0)
print ("Mallory decrypts the message from the ciphertext : ", decrypted_aes_mallory)

Alice encrypts the message 'Hello Bob', to the ciphertext:  b'k\xcc\xf2mS\x12_3W\xf3p\xfa\x00z\xc4\xc2'
Mallory decrypts the message from the ciphertext :  Hello Bob


### Repeat the Attack Using Generator a

In [ ]:
# Alice sends q and a to Bob
q = 37
a = 5

# But then Mallory changes a to 1: this means that all values will equal 1!
a = 1

# Alice generates private and public keys
priv_alice = secrets.randbelow(q) #Alice private
pub_alice = pow(a,priv_alice, q) #Alice public

# Bob generates private and public keys
priv_bob = secrets.randbelow(q) #Bob private
pub_bob = pow(a, priv_bob, q) #Bob public

print("Alice public value: ", pub_alice)
print("Bob public value: ", pub_bob)

# Alice and Bob create shared keys
shared_alice = pow(pub_bob, priv_alice, q)
shared_bob = pow(pub_alice, priv_bob, q)

# Mallory knows that all private and public keys equal 1, so she is able to 
# create her own shared valye
shared_mallory = pow(1, 1, q)

print("Alice's secret value:", shared_alice)
print("Bob's secret value:  ", shared_bob)

# Alice generates key
aes_key = derive_aes_key(shared_alice)
i_v = b'\x00' * 16  # shared initialization vector

# BUT MALLORY IS ABLE TO GENERATE KEY ALSO!
mallory_aes_key = derive_aes_key(shared_alice)

# alice encrypts message
c0 = aes_encrypt(aes_key, i_v, "Hello Bob")
print("Alice encrypts the message 'Hello Bob', to the ciphertext: ", c0)

# and mallory decrypts it
decrypted_aes_mallory = aes_decrypt(mallory_aes_key, i_v, c0)
print ("Mallory decrypts the message from the ciphertext : ", decrypted_aes_mallory)

Alice public value:  1
Bob public value:  1
Alice's secret value: 1
Bob's secret value:   1
Alice encrypts the message 'Hello Bob', to the ciphertext:  b'6\x93d\xb2uv\x06\xe7\xd8\xb6\xd6\x1a\x8e\xc2\xed\xc9'
Mallory decrypts the message from the ciphertext :  Hello Bob


## Task 3: Implement “Textbook” RSA & MITM Key Fixing via Malleability: 

### Part 1

In [ ]:
the_gcd = 0
e = 65537

# make sure that gcd of e, theta_n = 1 
while (the_gcd != 1):
    p = getPrime(1024)
    q = getPrime(1024)
    n = p * q
    theta_n = (p - 1) * (q - 1)
    the_gcd = gcd(e, theta_n)

# calculate d
d = pow(e, -1, theta_n)

# at this point, have public and private keys ...
public_key = (e, n)
private_key = (d, n)

In [56]:
# consumes string, returns integer representation
def text_to_int(text):
    hex_str = text.encode('ascii').hex()
    int_val = int(hex_str, 16)
    return int_val

# consumes int represenation, returns string 
def int_to_text(int_val):
    hex_str = hex(int_val)[2:]
    bytes_obj = bytes.fromhex(hex_str)
    return bytes_obj.decode('utf-8')

In [76]:
# rsa enncryption
# consumes text, returns encryption
def rsa_encrypt(text, public_key): 
    int_val = text_to_int(text)
    encrypted_msg = pow(int_val, public_key[0], public_key[1])
    return encrypted_msg

# rsa decryption
# consumes encrypted text, returns decrypted text
def rsa_decrypt(encrypted_msg, private_key):
    decrypted_msg = pow(encrypted_msg, private_key[0], private_key[1])
    return int_to_text(decrypted_msg)

In [77]:
# demo
encryption = rsa_encrypt("hey how are you doing?", public_key)
rsa_decrypt(encryption, private_key)

'hey how are you doing?'

In [78]:
encryption = rsa_encrypt("blah blah blah blah blah blah blah", public_key)
rsa_decrypt(encryption, private_key)

'blah blah blah blah blah blah blah'

### Part 2

In [82]:
# Alice calculates n and e
the_gcd = 0
e = 65537

while (the_gcd != 1):
    p = getPrime(1024)
    q = getPrime(1024)
    n = p * q
    theta_n = (p - 1) * (q - 1)
    the_gcd = gcd(e, theta_n)

# Alice calculates d
d = pow(e, -1, theta_n)

# Alice sends out public key and keeps private key
public_key = (e, n)
private_key = (d, n)

# Mallory is able to see public key, so she knows public keys

# Bob receives this public key and encrypts a message
bob_encryption = rsa_encrypt("Hi, I am Bob and I am sending this message!", public_key)

# Mallory intercepts this message and sends her own encrypted message to Alice

# first: Mallory finds s 
s = getPrime(1024)
mallory_gcd = 0
while (mallory_gcd != 1):
    mallory_gcd = gcd(s, n)
    s = getPrime(128)

# then: Mallory encrypts 
mallory_encryption = (bob_encryption * pow(s, public_key[0], public_key[1])) % public_key[1]

# Alice recieves Mallory's encrypted message and decrypts it 
alice_decryption = pow(mallory_encryption, private_key[0], private_key[1])

# Mallory can now do some math and decrypt messages
s_inv = pow(s, -1, n)
m = (alice_decryption * s_inv) % n
int_to_text(m)

'Hi, I am Bob and I am sending this message!'

In [ ]:
# Show how Mallory can create a valid signature for a third message, me = m1 * m2

m1 = text_to_int("hey hey hey")
m1_sign = pow(m1, d, n)

m2 = text_to_int("woah woah woah")
m2_sign = pow(m2, d, n)

m3_sign = (m1_sign * m2_sign) % n
pow(m3_sign, e, n)

305728935813309016048131619779895479542048240524426994717224

## Questions

1. It would be difficult for an adversary to solve the DHP with these parameters. An attacker could see:
- The public prime q
- The generator a 
- Alices public value A = a^x mod q
- Bobs public value B = a^y mod q

    Without knowledge about Alice's private key x or Bob's private key y, the only way you can compute the same AES key to decrypt a message, is if they find the shared secret k, computed by a^(xy) mod q. 

    The attacker knows the public key A = a^x mod q. The question becomes, can you find x from A? This is a math problem known as the Discrete Logarithm Proble,. x = log(a)A (mod q). "Given the base a, the modulus q, and the result A, find the exponent x". This is incredibly difficult for large numbers for q, and requires an unfeasible amount of computation. 

    Due to this mathematically infeasibility attackers most likely would not try and break it directly. Instead, they would most likely try to exploit weak links in the overall system. For example, a Man in the Middle attack, where an attacker tricks Alice or Bob into establishing keys with the attacker, thinking they are establishing keys with each other. 



2. The strategy used for the tiny parameters would not work for the large values of q and a. As we noted in the previous question, the mathematical strength of this implementation comes for very large numbers. If you had parameters like in our small example, q = 37, a = 5. An adversary could geuess a private key x, and compute a^x mod q to generate a public key. They would then compare with Alice's public key until they match. Since q is so small, there are realtively few possible values of x, and can be computed extremely quickly. This is why the IETF reccommends a 1024-bit prime for q, the math becomes stronger with larger numbers. 



3. These attacks are possible because Diffie-Hellman never verifies the identify of who is sending the message. To prevent the attacks it would be necessary to incorporate some sort of mechanism to track and verify the identity of who sends each message. 
4. If two people use the same value for n, it is possible for an attacker who knows e and n to calculate d. This would enable the attacker to decrypt all messages that are sent.

## Citations

[Diffie-Hellman reference 1](https://security.stackexchange.com/questions/91699/why-cant-i-mitm-a-diffie-hellman-key-exchange)

[Diffie-Hellman reference 2](https://stackoverflow.com/questions/10471009/how-does-the-man-in-the-middle-attack-work-in-diffie-hellman)

[Hex and Int type conversions](https://www.google.com/search?q=decode+hex+to+text+string+python&newwindow=1&sca_esv=51db692ab24d98dd&rlz=1C5CHFA_enUS1188US1188&sxsrf=ANbL-n5U-1ftdloDsB9OPB4uYt-jvZN3JQ%3A1770143680438&ei=wD-CaY-2Gtz4kPIPi5uguQ4&biw=1470&bih=797&aic=0&ved=0ahUKEwiPlpiK-72SAxVcPEQIHYsNKOcQ4dUDCBE&uact=5&oq=decode+hex+to+text+string+python&gs_lp=Egxnd3Mtd2l6LXNlcnAiIGRlY29kZSBoZXggdG8gdGV4dCBzdHJpbmcgcHl0aG9uMgkQABixBhgWGB4yDhAAGIAEGLEGGIYDGIoFMg4QABiABBixBhiGAxiKBTIOEAAYgAQYsQYYhgMYigUyCBAAGLEGGO8FMggQABixBhjvBTILEAAYgAQYsQYYogQyCxAAGLEGGKIEGIkFMgsQABiABBixBhiiBEj4B1DLAVjwBnABeAGQAQCYAY0BoAHABqoBAzAuN7gBA8gBAPgBAZgCCKAC5gbCAgoQABiwAxjWBBhHmAMAiAYBkAYIkgcDMS43oAfgKLIHAzAuN7gH2wbCBwUwLjMuNcgHGYAIAA&sclient=gws-wiz-serp)

ChatGPT was also used for simple debugging and discussion about syntax. Helped me to understand the Discrete Logarithm Problem for finding a private key from a public, as stated in Discussion Q 1. 
